# Step 7 - Redudancy Analysis of bicycle network results
## Project: Growing Urban Bicycle Networks with LTNs

This notebook takes the results of notebook 04 and finds the locations where LTNs would make any built cycle network redudant.

In [ ]:
# import libraries
from src import utils
PATH = utils.PATH # shortening the var name so that we don't have to change it below

# System
import csv
import os
import dill as pickle
import itertools
import random
from collections import defaultdict
import pprint
pp = pprint.PrettyPrinter(indent=4)
from tqdm.notebook import tqdm
import glob
from concurrent.futures import ThreadPoolExecutor
from copy import deepcopy
import yaml
import json

# Math/Data
import numpy as np
import pandas as pd


# Network
import networkx as nx

# Plotting
import matplotlib.pyplot as plt
import matplotlib.animation as animation


# Geo
import osmnx as ox
ox.settings.log_file = True
ox.settings.requests_timeout = 300
ox.settings.logs_folder = PATH["logs"]
import geopandas as gpd
import json
import folium

## Preliminaries

### Parameters

In [ ]:
debug = False # If True, will produce plots and/or verbose output to double-check

In [ ]:
params = yaml.load(
    open("../parameters/parameters.yml"), 
    Loader=yaml.FullLoader)
osmnxparameters = json.load(open("../parameters/osmnxparameters.json", "r"))
plotparam = json.load(open("../parameters/plotparam.json", "r"))
plotparam_analysis = json.load(open("../parameters/plotparam_analysis.json", "r"))

### Load Cities

In [ ]:
# load cities
cities = utils.load_cities(PATH, debug)

## Loading

### Load Results

In [ ]:
# betweenness 
betweenness_results = {}
for scenario in params["scenarios"]:
    betweenness_results[scenario] = {}
    for placeid in cities:
        filename = (PATH["results"] + placeid + "/" + scenario + "/" + f"{placeid}_poi_{params['poi_source']}_betweenness_weighted_" + scenario + ".pickle")
        abs_path = os.path.abspath(filename)
        if os.path.exists(abs_path):
            with open(abs_path, "rb") as f:
                betweenness_results[scenario][placeid] = pickle.load(f)
        else:
            print(f"File {abs_path} does not exist.")
            print("Please run the betweenness analysis first.")
            print(f"No betweenness files found for {placeid} in scenario {scenario}.")

In [ ]:
# random (many runs to get a distribution)
random_results = {}
for scenario in params["scenarios"]:
    random_results[scenario] = {}
    for placeid in cities:
        pattern = (PATH["results"] + placeid + "/" + scenario + "/" +
                   f"{placeid}_poi_{params['poi_source']}_random_weighted_{scenario}_run*.pickle")
        random_files = sorted(glob.glob(os.path.abspath(pattern)))#[:3]  # only take the first 3 whilst we debug :D
        if random_files:
            random_results[scenario][placeid] = []
            for fn in random_files:
                abs_path = os.path.abspath(fn)
                with open(abs_path, "rb") as f:
                    res = pickle.load(f)
                random_results[scenario][placeid].append(res)
        else:
            print(f"No random files found for {placeid} in scenario {scenario}.")
            print("Please run the random growth analysis first.")


In [ ]:
# demand
demand_results = {}
for scenario in params["scenarios"]:
    demand_results[scenario] = {}
    for placeid in cities:
        filename = (PATH["results"] + placeid + "/" + scenario + "/" + f"{placeid}_poi_{params['poi_source']}_demand_weighted_" + scenario + ".pickle")
        abs_path = os.path.abspath(filename)
        if os.path.exists(abs_path):
            with open(abs_path, "rb") as f:
                demand_results[scenario][placeid] = pickle.load(f)
        else:
            print(f"File {abs_path} does not exist.")
            print("Please run the demand analysis first.")
            print(f"No demand files found for {placeid} in scenario {scenario}.")


In [ ]:
# demand LTN priority
demand_ltn_priority_results = {}
for scenario in params["scenarios"]:
    demand_ltn_priority_results[scenario] = {}
    for placeid in cities:
        filename = (PATH["results"] + placeid + "/" + scenario + "/" + f"{placeid}_poi_{params['poi_source']}_demand_ltn_priority_weighted_" + scenario + ".pickle")
        abs_path = os.path.abspath(filename)
        if os.path.exists(abs_path):
            with open(abs_path, "rb") as f:
                demand_ltn_priority_results[scenario][placeid] = pickle.load(f)
        else:
            print(f"File {abs_path} does not exist.")
            print("Please run the demand LTN priority analysis first.")
            print(f"No demand LTN priority files found for {placeid} in scenario {scenario}.")


In [ ]:
# betweenness LTN priority
betweenness_ltn_priority_results = {}
for scenario in params["scenarios"]:
    betweenness_ltn_priority_results[scenario] = {}
    for placeid in cities:
        filename = (PATH["results"] + placeid + "/" + scenario + "/" + f"{placeid}_poi_{params['poi_source']}_betweenness_ltn_priority_weighted_" + scenario + ".pickle")
        abs_path = os.path.abspath(filename)
        if os.path.exists(abs_path):
            with open(abs_path, "rb") as f:
                betweenness_ltn_priority_results[scenario][placeid] = pickle.load(f)
        else:
            print(f"File {abs_path} does not exist.")
            print("Please run the betweenness LTN priority analysis first.")
            print(f"No betweenness LTN priority files found for {placeid} in scenario {scenario}.")


Find investment level, split results into GTs, GT_abstracts 

In [ ]:
for scenario_name in params["scenarios"]:
    for placeid in cities:
        # Demand 
        if placeid in demand_results.get(scenario_name, {}):
            demand_dict = demand_results[scenario_name][placeid]
            investment_levels_demand = demand_dict["prune_quantiles"]
            GTs_demand               = demand_dict["GTs"]
            GT_abstracts_demand      = demand_dict["GT_abstracts"]
        else:
            print(f"No demand results for {placeid} in scenario '{scenario_name}'")
            investment_levels_demand = []
            GTs_demand               = []
            GT_abstracts_demand      = []


        # Betweenness‐LTN‐priority 
        if placeid in betweenness_ltn_priority_results.get(scenario_name, {}):
            betweenness_ltn_dict = betweenness_ltn_priority_results[scenario_name][placeid]
            investment_levels_betw = betweenness_ltn_dict["prune_quantiles"]
            GTs_betw               = betweenness_ltn_dict["GTs"]
            GT_abstracts_betw      = betweenness_ltn_dict["GT_abstracts"]
        else:
            # e.g. scenario == "no_ltn_scenario" has no betweenness‐LTN‐priority data
            investment_levels_betw = []
            GTs_betw               = []
            GT_abstracts_betw      = []

        # Betweenness
        if placeid in betweenness_results.get(scenario_name, {}):
            betweenness_dict = betweenness_results[scenario_name][placeid]
            investment_levels_betweenness = betweenness_dict["prune_quantiles"]
            GTs_betweenness               = betweenness_dict["GTs"]
            GT_abstracts_betweenness      = betweenness_dict["GT_abstracts"]
        else:
            investment_levels_betweenness = []
            GTs_betweenness               = []
            GT_abstracts_betweenness      = []

        # Demand‐LTN‐priority 
        if placeid in demand_ltn_priority_results.get(scenario_name, {}):
            dem_ltn_dict = demand_ltn_priority_results[scenario_name][placeid]
            investment_levels_dem_ltn = dem_ltn_dict["prune_quantiles"]
            GTs_dem_ltn               = dem_ltn_dict["GTs"]
            GT_abstracts_dem_ltn      = dem_ltn_dict["GT_abstracts"]
        else:
            investment_levels_dem_ltn = []
            GTs_dem_ltn               = []
            GT_abstracts_dem_ltn      = []

        # Random‐runs (loads all run*.pickle files)
        random_runs_list = random_results.get(scenario_name, {}).get(placeid, [])
        if random_runs_list:
            all_GTs_random       = [run_dict["GTs"]          for run_dict in random_runs_list]
            all_GTabs_random      = [run_dict["GT_abstracts"]  for run_dict in random_runs_list]
            investment_levels_random = random_runs_list[0]["prune_quantiles"]
        else:
            all_GTs_random          = []
            all_GTabs_random         = []
            investment_levels_random = []



### Load existing networks, nodes, GeoDataframe



In [ ]:
G_biketracks_dict               = {}  # (placeid, scenario) → biketrack graph
G_biketrack_no_ltns_dict       = {}  # (placeid, scenario) → biketrack_no_ltn graph
G_biketrackcaralls_dict        = {}  # (placeid, scenario) → biketrackcarall graph
G_biketrackcarall_edges_dict    = {}  # (placeid, scenario) → GeoDataFrame of biketrackcarall edges
boundary_gdfs               = {}  # placeid → boundary GeoDataFrame (same for all scenarios)
tess_points_dict            = {}  # (placeid, scenario) → tessellation points GeoDataFrame
ltn_points_dict             = {}  # (placeid, scenario) → LTN points GeoDataFrame
combined_points_dict        = {}  # (placeid, scenario) → combined points GeoDataFrame

for scenario in params["scenarios"]:
    for placeid, placeinfo in cities.items():
        base_folder = os.path.join(PATH["data"], placeid, scenario)

        # Load biketrack graph
        biketrack_gpkg = os.path.join(base_folder, f"{placeid}_biketrack.gpkg")
        if os.path.exists(biketrack_gpkg):
            G_biketrack = utils.ox_gpkg_to_graph(biketrack_gpkg)
            G_biketrack.remove_nodes_from(list(nx.isolates(G_biketrack)))
            G_biketracks_dict[(placeid, scenario)] = G_biketrack
        else:
            print(f"Missing: {biketrack_gpkg}")
            G_biketracks_dict[(placeid, scenario)] = None

        # Load biketrack_no_ltn graph
        biketrack_no_ltn_gpkg = os.path.join(base_folder, f"{placeid}_biketrack_no_ltn.gpkg")
        if os.path.exists(biketrack_no_ltn_gpkg):
            G_no_ltn = utils.ox_gpkg_to_graph(biketrack_no_ltn_gpkg)
            G_no_ltn.remove_nodes_from(list(nx.isolates(G_no_ltn)))
            G_biketrack_no_ltns_dict[(placeid, scenario)] = G_no_ltn
        else:
            print(f"Missing: {biketrack_no_ltn_gpkg}")
            G_biketrack_no_ltns_dict[(placeid, scenario)] = None

        # Load biketrackcarall graph
        biketrackcarall_gpkg = os.path.join(base_folder, f"{placeid}_biketrackcarall.gpkg")
        if os.path.exists(biketrackcarall_gpkg):
            G_carall = utils.ox_gpkg_to_graph(biketrackcarall_gpkg)
            G_carall.remove_nodes_from(list(nx.isolates(G_carall)))
            G_biketrackcaralls_dict[(placeid, scenario)] = G_carall

            # also store edges GeoDataFrame
            edges_gdf = ox.graph_to_gdfs(G_carall, nodes=False)
            G_biketrackcarall_edges_dict[(placeid, scenario)] = edges_gdf
        else:
            print(f"Missing: {biketrackcarall_gpkg}")
            G_biketrackcaralls_dict[(placeid, scenario)] = None
            G_biketrackcarall_edges_dict[(placeid, scenario)] = None

        #  Load boundary once per placeid (it won’t change by scenario)
        if placeid not in boundary_gdfs:
            boundary_gdf = ox.geocode_to_gdf(placeinfo["nominatimstring"])
            boundary_gdfs[placeid] = boundary_gdf

        # get nodes
        tess_points_gpkg = os.path.join(base_folder, f"{placeid}_tessellation_points.gpkg")
        if os.path.exists(tess_points_gpkg):
            tess_points = gpd.read_file(tess_points_gpkg)
            tess_points_dict[(placeid, scenario)] = tess_points
        else:
            print(f"Missing: {tess_points_gpkg}")
            tess_points_dict[(placeid, scenario)] = None
        
        # get ltn points
        if scenario != "no_ltn_scenario":
            ltn_points_gpkg = os.path.join(base_folder, f"{placeid}_ltn_points.gpkg")
            if os.path.exists(ltn_points_gpkg):
                ltn_points = gpd.read_file(ltn_points_gpkg)
                ltn_points_dict[(placeid, scenario)] = ltn_points
            else:
                print(f"Missing: {ltn_points_gpkg}")
                ltn_points_dict[(placeid, scenario)] = None
        
        # get combined points
        combined_points_gpkg = os.path.join(base_folder, f"{placeid}_combined_points.gpkg")
        if os.path.exists(combined_points_gpkg):
            combined_points = gpd.read_file(combined_points_gpkg)
            combined_points_dict[(placeid, scenario)] = combined_points
        else:
            print(f"Missing: {combined_points_gpkg}")
            combined_points_dict[(placeid, scenario)] = None

        # get all neighbourhoods (ragardless of their low traffic status. This doesn't change by scenario)
        all_neighbourhoods = gpd.read_file(PATH["data"] + placeid + "/" + 'neighbourhoods_'+  placeid + '.gpkg')
        all_neighbourhoods_centroids = all_neighbourhoods.geometry.centroid
        all_neighbourhoods_centroids = gpd.GeoDataFrame(geometry= all_neighbourhoods_centroids, crs=all_neighbourhoods.crs)



# Analyse - By Neighbourhood Boundary

In [ ]:
# Load Neighbourhoods
LTNs_dict = {}

for scenario in params["scenarios"]:
    for placeid, placeinfo in cities.items():
        fname = f"scored_neighbourhoods_{placeid}.gpkg"
        fp = os.path.join(PATH["data"], placeid, scenario, fname)
        if os.path.exists(fp):
            try:
                LTNs_dict[(placeid, scenario)] = gpd.read_file(fp)
            except Exception as e:
                print(f"Error reading {fp}: {e}")
                LTNs_dict[(placeid, scenario)] = None
        else:
            print(f"Missing: {fp}")
            LTNs_dict[(placeid, scenario)] = None
            print("Missing no LTN scenario neighbourhoods is expected (it doesn't haven any to get...)")

if debug:
    for (placeid, scen), gdf in LTNs_dict.items():
        if gdf is None or gdf.empty: 
            continue
        ax = gdf.plot(figsize=(6,6), linewidth=0.5)
        ax.set_title(f"{placeid} — {scen}")
        ax.set_axis_off()
        plt.show()

In [ ]:
# Extract final networks from all analysis types for each scenario
analysis_types = ['demand', 'betweenness', 'demand_ltn_priority', 'betweenness_ltn_priority']
result_dicts = {'demand': demand_results,'betweenness': betweenness_results,  'demand_ltn_priority': demand_ltn_priority_results, 'betweenness_ltn_priority': betweenness_ltn_priority_results}

# Dictionary to store the 100th iteration network
final_grown_networks = {}

for analysis_type in analysis_types:
    final_grown_networks[analysis_type] = {}
    for scenario in params["scenarios"]:
        results_dict = result_dicts[analysis_type].get(scenario, {}).get(placeid) 
        if results_dict and results_dict.get("GTs"):
            final_network = results_dict["GTs"][99]  # Get the final (most grown) network
            final_grown_networks[analysis_type][scenario] = final_network
            #print(f"{analysis_type} - {scenario}: {final_network.number_of_nodes()} nodes, {final_network.number_of_edges()} edges")
        else:
            final_grown_networks[analysis_type][scenario] = None
            print(f"{analysis_type} - {scenario}: No network found")


In [ ]:
pairs = [("no_ltn_scenario","current_ltn_scenario"),
         ("no_ltn_scenario","more_ltn_scenario"),
         ("current_ltn_scenario","more_ltn_scenario")]


for placeid in cities:
    edges = {s: ox.graph_to_gdfs(G)[1].to_crs("EPSG:27700") 
             for s, G in final_grown_networks["demand"].items() if G is not None}
    for scen in edges: edges[scen]["length"] = edges[scen].geometry.length
    redundancy_results = [] 
    for net_scen, ltn_scen in pairs:
        e = edges.get(net_scen)
        ltn = LTNs_dict.get((placeid, ltn_scen))
        if e is None or ltn is None or ltn.empty:
            continue
        ltn = ltn.to_crs("EPSG:27700")
        ltn_buffered = ltn.buffer(-5) # avoid bounding roads
        ltn_buffered = ltn_buffered[~ltn_buffered.is_empty & ltn_buffered.is_valid]
        ltn_union = ltn_buffered.unary_union

        # Calculate redundancy
        inside_mask = e.geometry.within(ltn_union)
        inside_edges = e[inside_mask]
        inside_len = inside_edges.geometry.length.sum()
        total_len = e["length"].sum()

        print(f"{placeid} | {net_scen} in {ltn_scen}: "
              f"{inside_len:.1f} m ({inside_len/total_len*100:.1f}%), "
              f"{len(inside_edges)}/{len(e)} edges")

        # Save results in memory
        redundancy_results.append({
            "placeid": placeid,
            "network_scenario": net_scen,
            "ltn_scenario": ltn_scen,
            "inside_length_m": inside_len,
            "total_length_m": total_len,
            "percentage_inside": inside_len / total_len * 100 if total_len > 0 else 0,
            "edges_inside": len(inside_edges),
            "total_edges": len(e)})

        print(f"{placeid} | {net_scen} in {ltn_scen}: "
              f"{inside_len:.1f} m ({inside_len / total_len * 100:.1f}%), "
              f"{len(inside_edges)}/{len(e)} edges")

    # Save 
    if redundancy_results:
        results_dir = os.path.join(PATH["results"], placeid)
        os.makedirs(results_dir, exist_ok=True)
        df = pd.DataFrame(redundancy_results)
        csv_fp = os.path.join(results_dir, f"redundancy_results_{placeid}.csv")
        pickle_fp = os.path.join(results_dir, f"redundancy_results_{placeid}.pkl")
        df.to_csv(csv_fp, index=False)
        with open(pickle_fp, "wb") as f:
            pickle.dump(redundancy_results, f)

In [ ]:
pairs = [("no_ltn_scenario","current_ltn_scenario"),
         ("no_ltn_scenario","more_ltn_scenario"),
         ("current_ltn_scenario","more_ltn_scenario")]

for placeid in cities:
    edges = {s: ox.graph_to_gdfs(G)[1].to_crs("EPSG:3857") 
             for s, G in final_grown_networks["demand"].items() if G is not None}
    for scen in edges: edges[scen]["length"] = edges[scen].geometry.length

    for net_scen, ltn_scen in pairs:
        e = edges.get(net_scen)
        ltn = LTNs_dict.get((placeid, ltn_scen))
        if e is None or ltn is None or ltn.empty: 
            continue
        ltn = ltn.to_crs("EPSG:3857")
        ltn_buffered = ltn.buffer(-5)  # same shrink as analysis
        ltn_buffered = ltn_buffered[~ltn_buffered.is_empty & ltn_buffered.is_valid]
        if ltn_buffered.empty:
            continue
        ltn_union = ltn_buffered.unary_union

        # Use 'within' to match analysis
        inside_edges = e[e.geometry.within(ltn_union)]

        # Convert all to WGS84 for Folium
        e_wgs   = e.to_crs("EPSG:4326")
        ltn_wgs = ltn.to_crs("EPSG:4326")
        red_wgs = inside_edges.to_crs("EPSG:4326")

        # Create map centered on the network
        center = e_wgs.geometry.unary_union.centroid.coords[0]
        m = folium.Map(location=[center[1], center[0]], zoom_start=13, tiles="CartoDB positron")

        folium.GeoJson(ltn_wgs, style_function=lambda x: {"color": "lightblue", "fill": True, "weight": 2}).add_to(m)
        folium.GeoJson(e_wgs, style_function=lambda x: {"color": "green", "weight": 1.5}).add_to(m)
        folium.GeoJson(red_wgs, style_function=lambda x: {"color": "red", "weight": 3}).add_to(m)

        if debug:
            display(m)
        m.save(PATH["plots"] + "overall_results" + "/" + f"redundancy_map_{placeid}_{net_scen}_in_{ltn_scen}.html")

In [ ]:
places = ["newcastle", "south_tyneside", "north_tyneside", "sunderland", "gateshead"]
results = {}
for placeid in places:
    pickle_fp = os.path.join(PATH["results"], placeid, f"redundancy_results_{placeid}.pkl")
    if os.path.exists(pickle_fp):
        with open(pickle_fp, "rb") as f:
            results[placeid] = pickle.load(f)
    else:
        print(f"Warning: no results found for {placeid}. Please run the above code for all the locations first.")

all_results = []

for place in places:
    fp = os.path.join(PATH["results"], place, f"redundancy_results_{place}.csv")
    df = pd.read_csv(fp)
    df["placeid"] = place  # Add placeid as a column
    all_results.append(df)

# Combine 
combined_df = pd.concat(all_results, ignore_index=True)
num_cols = combined_df.select_dtypes(include="number").columns # find just the numeric columns
mean_df = combined_df.groupby(["network_scenario", "ltn_scenario"])[num_cols].mean().reset_index()
range_df = combined_df.groupby(["network_scenario", "ltn_scenario"])[num_cols].agg(lambda x: x.max() - x.min()).reset_index()


if debug:
    print("Mean across places:")
    print(mean_df)
    print("\nRange across places:")
    print(range_df)


for (net_scen, ltn_scen), group in combined_df.groupby(["network_scenario", "ltn_scenario"]):
    print(f"\nScenario: {net_scen} in {ltn_scen}")
    for col in num_cols:
        col_mean = group[col].mean()
        col_min = group[col].min()
        col_max = group[col].max()
        col_range = col_max - col_min
        print(f"{col}: mean={col_mean:.1f}, min={col_min:.1f}, max={col_max:.1f}, range={col_range:.1f}")

In [ ]:
# Print total length of the full bike network (biketrackcarall edges) per place and scenario
scenario_totals = {s: 0.0 for s in params["scenarios"]}
scenario_counts = {s: 0 for s in params["scenarios"]}

for placeid in cities:
    for scenario in params["scenarios"]:
        edges_gdf = G_biketrackcarall_edges_dict.get((placeid, scenario))
        if edges_gdf is None or edges_gdf.empty:
            print(f"{placeid} | {scenario}: No biketrackcarall edges found")
            continue

        # project to metric CRS for accurate length measurement (use Web Mercator)
        edges_m = edges_gdf.to_crs("EPSG:3857")
        total_len_m = edges_m.geometry.length.sum()

        print(f"{placeid} | {scenario}: total bike network length = {total_len_m:.1f} m ({total_len_m/1000:.2f} km)")

        scenario_totals[scenario] += total_len_m
        scenario_counts[scenario] += 1

# Print aggregated totals (sum across places) and average per place for each scenario
print("\nAggregated totals by scenario:")
for scenario in params["scenarios"]:
    total = scenario_totals[scenario]
    count = scenario_counts[scenario]
    if count == 0:
        print(f"{scenario}: no data")
    else:
        avg = total / count
        print(f"{scenario}: total = {total:.1f} m ({total/1000:.2f} km), average per place = {avg:.1f} m ({avg/1000:.2f} km)")